In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

print("Dataset berhasil dibaca!")

Dataset berhasil dibaca!


In [3]:
ratings_clean = ratings.copy()
movies_clean = movies.copy()

print("Salinan data untuk proses preparation berhasil dibuat!")

Salinan data untuk proses preparation berhasil dibuat!


In [5]:
pemeriksaan_data = pd.DataFrame({
    "Pemeriksaan": [
        "Jumlah baris ratings",
        "Jumlah baris movies",
        "Missing value pada ratings",
        "Missing value pada movies",
        "Duplikat seluruh baris pada ratings",
        "Duplikat seluruh baris pada movies",
        "Duplikat pasangan userId-movieId",
        "Duplikat movieId pada movies"
    ],
    "Hasil": [
       len(ratings_clean),
       len(movies_clean),
       ratings_clean.isnull().sum().sum(),
       movies_clean.isnull().sum().sum(),
       ratings_clean.duplicated().sum(),
       movies_clean.duplicated().sum(),
       ratings_clean.duplicated(subset=["userId", "movieId"]).sum(),
       movies_clean.duplicated(subset=["movieId"]).sum()
    ]
})
display(pemeriksaan_data)

,Pemeriksaan,Hasil
0,Jumlah baris ratings,100836
1,Jumlah baris movies,9742
2,Missing value pada ratings,0
3,Missing value pada movies,0
4,Duplikat seluruh baris pada ratings,0
5,Duplikat seluruh baris pada movies,0
6,Duplikat pasangan userId-movieId,0
7,Duplikat movieId pada movies,0


In [6]:
ratings_clean = ratings_clean.dropna(
    subset=["userId", "movieId", "rating"]
)

movies_clean = movies_clean.dropna(
    subset=["movieId", "title", "genres"]
)

ratings_clean = ratings_clean.sort_values("timestamp").drop_duplicates(
    subset=["userId", "movieId"],
    keep="last"
)

movies_clean = movies_clean.drop_duplicates(
    subset=["movieId"],
    keep="first"
)

ratings_clean = ratings_clean[
    ratings_clean["rating"].between(0.5, 5.0)
]

print("Proses cleaning selesai!!")

Proses cleaning selesai!!


In [7]:
perbandingan_cleaning = pd.DataFrame({
    "Dataset": ["ratings", "movies"],
    "sebelum cleaning": [
        len(ratings),
        len(movies)
    ],
    "sesudah cleaning": [
        len(ratings_clean),
        len(movies_clean)
    ]
})

perbandingan_cleaning["Data Berkurang"] = (
    perbandingan_cleaning["sebelum cleaning"]
    - perbandingan_cleaning["sesudah cleaning"]
)

display(perbandingan_cleaning)

,Dataset,sebelum cleaning,sesudah cleaning,Data Berkurang
0,ratings,100836,100836,0
1,movies,9742,9742,0


In [9]:
ratings_movies = ratings_clean.merge(
    movies_clean[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

print("Data ratings dan movies berhasil digabungkan!")
display(ratings_movies.head(10))

Data ratings dan movies berhasil digabungkan!


,userId,movieId,rating,timestamp,title,genres
0,429,165,4.0,828124615,Die Hard: With a Vengeance (1995),Action|Crime|Thriller
1,429,595,5.0,828124615,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX
2,429,434,4.0,828124615,Cliffhanger (1993),Action|Adventure|Thriller
3,429,590,5.0,828124615,Dances with Wolves (1990),Adventure|Drama|Western
4,429,588,5.0,828124615,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
5,429,349,3.0,828124615,Clear and Present Danger (1994),Action|Crime|Drama|Thriller
6,429,421,4.0,828124615,Black Beauty (1994),Adventure|Children|Drama
7,429,420,2.0,828124615,Beverly Hills Cop III (1994),Action|Comedy|Crime|Thriller
8,429,592,5.0,828124615,Batman (1989),Action|Crime|Thriller
9,429,351,4.0,828124615,"Corrina, Corrina (1994)",Comedy|Drama|Romance


In [10]:
cek_judul_kosong = pd.DataFrame({
    "Pemeriksaan": [
        "Jumlah total rating setelah penggabungan",
        "Jumlah rating tanpa judul film",
        "Jumlah rating dengan judul film"
    ],
    "Hasil": [
        len(ratings_movies),
        ratings_movies["title"].isnull().sum(),
        ratings_movies["title"].notnull().sum()
    ]
})

display(cek_judul_kosong)

,Pemeriksaan,Hasil
0,Jumlah total rating setelah penggabungan,100836
1,Jumlah rating tanpa judul film,0
2,Jumlah rating dengan judul film,100836


In [11]:
ratings_model = ratings_clean[
    ["userId", "movieId", "rating"]
].copy()

print("Data khusus untuk model berhasil dibuat")
display(ratings_model.head(10))

Data khusus untuk model berhasil dibuat


,userId,movieId,rating
66669,429,165,4.0
66719,429,595,5.0
66713,429,434,4.0
66717,429,590,5.0
66716,429,588,5.0
66706,429,349,3.0
66711,429,421,4.0
66710,429,420,2.0
66718,429,592,5.0
66707,429,351,4.0


In [12]:
import numpy as np

print("NumPy berhasil diimpor")

NumPy berhasil diimpor


In [13]:
TEST_SIZE = 0.20
RANDOM_STATE = 42
MIN_RATINGS_PER_USER = 5

print("Parameter pembagian data berhasil ditetapkan")

Parameter pembagian data berhasil ditetapkan


In [14]:
jumlah_rating_per_user = ratings_model.groupby("userId").size()

user_layak_split = jumlah_rating_per_user[
    jumlah_rating_per_user >= MIN_RATINGS_PER_USER
].index

ratings_split = ratings_model[
    ratings_model["userId"].isin(user_layak_split)
].copy()

ringkasan_user_split = pd.DataFrame({
    "Informasi": [
        "Jumlah user awal",
        "Jumlah user layak dibagi",
        "Jumlah user tidak layak dibagi",
        "Jumlah rating yang digunakan"
    ],
    "Nilai": [
        ratings_model["userId"].nunique(),
        len(user_layak_split),
        ratings_model["userId"].nunique() - len(user_layak_split),
        len(ratings_split)
    ]
})

display(ringkasan_user_split)

,Informasi,Nilai
0,Jumlah user awal,610
1,Jumlah user layak dibagi,610
2,Jumlah user tidak layak dibagi,0
3,Jumlah rating yang digunakan,100836


In [16]:
from sklearn.model_selection import train_test_split

train_list = []
test_list = []

for user_id, data_user in ratings_model.groupby("userId"):
    train_user, test_user = train_test_split(
        data_user,
        test_size=0.2,
        random_state=42
    )

    train_list.append(train_user)
    test_list.append(test_user)

train_data = pd.concat(train_list)
test_data = pd.concat(test_list)

print("Pembagian train dan test selesai")

Pembagian train dan test selesai


In [17]:
ringkasan_split = pd.DataFrame({
    "Data": ["Train", "Test", "Total"],
    "Jumlah Rating": [
        len(train_data),
        len(test_data),
        len(train_data) + len(test_data)
    ]
})

ringkasan_split["Persentase (%)"] = (
    ringkasan_split["Jumlah Rating"] / len(ratings_model) * 100
).round(2)

display(ringkasan_split)

,Data,Jumlah Rating,Persentase (%)
0,Train,80419,79.75
1,Test,20417,20.25
2,Total,100836,100.00


In [18]:
matriks_user_film = train_data.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print("Matriks user-film berhasil dibuat.")
print("Ukuran matriks:", matriks_user_film.shape)

display(matriks_user_film.iloc[:5, :8])

Matriks user-film berhasil dibuat.
Ukuran matriks: (610, 8957)


movieId,1,2,3,4,5,6,7,8
userId,,,,,,,,
1,NaN,NaN,4.0,NaN,NaN,4.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
referensi_film = movies_clean[[
    "movieId",
    "title",
    "genres"
]].copy()

display(referensi_film.head(10))

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [ ]:
train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)
referensi_film.to_csv("referensi_film.csv", index=False)

print("File hasil preparation berhasil disimpan.")

File hasil preparation berhasil disimpan.
